# Facility location lab: capacitated p-median
Choose exactly p facilities while assigning every demand zone to a selected facility without exceeding site capacity. This is an exact tiny-instance teaching model, not a city-scale solver.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/p-median'))
sys.path.insert(0, str(ROOT / 'implementations/python/capacitated-p-median'))
from p_median import solve_p_median
from capacitated_p_median import solve_capacitated_p_median


## 1. Define demand, candidates and operational capacity
Rows are demand zones; columns are candidate facilities. Demand quantities consume capacity, while optional demand weights affect the objective. Keep those meanings separate.

In [ ]:
demand_xy = np.array([[0, 0], [2, 1], [5, 0], [1, 5]])
candidate_xy = np.array([[0, 1], [3, 0], [6, 1]])
costs = np.linalg.norm(demand_xy[:, None, :] - candidate_xy[None, :, :], axis=2)
demand = np.array([2, 1, 2, 1], dtype=float)
capacity = np.array([2, 2, 4], dtype=float)
print('Cost matrix:\n', np.round(costs, 2))
print('Demand quantities:', demand)
print('Candidate capacities:', capacity)


## 2. Solve the uncapacitated p-median first
The baseline is free to send any amount of demand to the cheapest open site. Inspect its assignments and calculate the resulting loads.

In [ ]:
plain_sites, plain_cost, plain_assignment = solve_p_median(costs, p=2)
plain_loads = np.array([demand[np.array(plain_assignment) == j].sum() for j in plain_sites])
print('Opened sites:', plain_sites)
print('Objective:', round(plain_cost, 3))
print('Assignment:', plain_assignment)
print('Loads at opened sites:', plain_loads)
print('Capacity exceeded:', bool(np.any(plain_loads > capacity[list(plain_sites)])))


## 3. Enforce capacity
Now each demand row must be assigned whole to one open site, and the sum of demand quantities assigned to a facility cannot exceed its capacity.

In [ ]:
sites, objective, assignment = solve_capacitated_p_median(
    costs, p=2, capacities=capacity, demand_quantities=demand
)
loads = np.array([demand[np.array(assignment) == j].sum() for j in sites])
print('Opened sites:', sites)
print('Objective:', round(objective, 3))
print('Assignment:', assignment)
print('Loads:', loads)
print('Capacities:', capacity[list(sites)])
assert np.all(loads <= capacity[list(sites)] + 1e-10)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for i, site in enumerate(assignment):
    ax.plot([demand_xy[i, 0], candidate_xy[site, 0]], [demand_xy[i, 1], candidate_xy[site, 1]], alpha=0.6)
ax.scatter(demand_xy[:, 0], demand_xy[:, 1], s=50 + 35 * demand, marker='o', label='Demand zones')
ax.scatter(candidate_xy[:, 0], candidate_xy[:, 1], s=130, marker='s', label='Candidate facilities')
ax.scatter(candidate_xy[list(sites), 0], candidate_xy[list(sites), 1], s=220, facecolors='none', edgecolors='black', linewidths=2, label='Opened facilities')
for i, (x, y) in enumerate(candidate_xy): ax.annotate(f'F{i} cap={capacity[i]:g}', (x, y), xytext=(5, 6), textcoords='offset points')
ax.set_title('Capacitated p-median assignment'); ax.set_aspect('equal'); ax.grid(True); ax.legend(); plt.show()


## 4. Test a capacity change
Reduce the capacity of the third candidate. Does the selected pair or assignment change? Then try capacities whose total is below total demand and inspect the explicit infeasibility response.

In [ ]:
reduced = capacity.copy(); reduced[2] = 2
try:
    changed = solve_capacitated_p_median(costs, 2, capacities=reduced, demand_quantities=demand)
    print('With reduced capacity:', changed)
except ValueError as exc:
    print('No feasible solution:', exc)
try:
    solve_capacitated_p_median(costs, 2, capacities=[1, 1, 1], demand_quantities=demand)
except ValueError as exc:
    print('Expected infeasibility:', exc)


## 5. Interpretation and limits
- Capacity and demand quantities must use the same unit and time period.
- The model treats each demand zone as indivisible; it cannot split a zone between sites.
- Capacity is a hard aggregate limit, not a queueing or service-quality model.
- This exhaustive solver is intended for tiny examples. Larger problems need a mathematical-programming solver or a suitable heuristic, validated against small exact cases.
- Try changing demand weights while keeping quantities fixed. Objective importance changes; physical load does not.